# Retail Intelligence

Customer & Sales Analytics using UCI Online Retail.

## 1. Imports and data loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np
df = pd.read_excel("Online_Retail.xlsx")

## 2. Exploring and cleaning the data

In [ ]:
#print(df.dtypes)
#print(df.shape)
#print(df.isna().sum())
#print(df.nunique())
#print(df.describe())
#print(df.head(10))

#print((df["Quantity"] < 0).sum())
#print((df["Quantity"] == 0).sum())
#print((df["UnitPrice"] < 0).sum())
#print((df["UnitPrice"]  == 0).sum())
#print(df[df["Quantity"] < 0].head(20))

In [ ]:
#print(df[df["InvoiceNo"].str.startswith("C", na=False)].head(20))
#print((df["InvoiceNo"].str.startswith("C",na=False)).sum())
#print(((df["InvoiceNo"].str.startswith("C",na=False)) & (df["Quantity"] < 0)).sum())
df[(df["Quantity"] < 0) & (~df["InvoiceNo"].str.startswith("C", na=False))]
sales_df = df[df["Quantity"] > 0].copy()
df[df["UnitPrice"] == 0].head(20)
df[df["CustomerID"].isna()].head(20)
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

## 3. Revenue analysis

In [ ]:
sales_df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
#print(sales_df.shape)
#print((sales_df["Quantity"] <= 0).sum())
#print((sales_df["UnitPrice"] <= 0).sum())

sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]
#print(sales_df.head())
print(sales_df["Revenue"].sum())

#print(sales_df["CustomerID"].nunique())
#print(sales_df["InvoiceNo"].nunique())
#print(sales_df.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10))
#print(sales_df["Description"].nunique())
#rint(sales_df["Description"].value_counts().head(20))
#print(sales_df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10))


top_products = sales_df.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
#top_products.plot(kind="bar")
#plt.show()

#print("\n",sales_df[sales_df["Description"].isin(
#    ["DOTCOM POSTAGE", "POSTAGE", "Manual"]
#)].shape)


## 4. Product analysis

In [ ]:
product_df = sales_df[~sales_df["Description"].isin(
    ["DOTCOM POSTAGE", "POSTAGE", "Manual"]
)].copy()


top_products = product_df.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
top_products

In [ ]:
top_products.plot(kind="bar")
plt.ylabel("Revenue")
plt.xlabel("Product")
plt.title("Top 10 Products by Revenue")
plt.show()


## 5. Monthly revenue

In [ ]:
sales_df["Month"] = sales_df["InvoiceDate"].dt.month
print(sales_df["Month"].unique())
monthly_revenue = sales_df.groupby("Month")["Revenue"].sum()
print(monthly_revenue)
monthly_revenue.plot(kind="bar")
plt.ylabel("Revenue")
plt.xlabel("Month")
plt.title("Revenue by month")
plt.show()

## 6. Customer revenue

In [ ]:
customer_revenue = product_df.groupby("CustomerID")["Revenue"].sum().sort_values(ascending=False).head(10)
print(customer_revenue)

customer_revenue.plot(kind="bar")
plt.ylabel("Revenue")
plt.xlabel("CustomerID")
plt.title("Top 10 customers")
plt.show()

## 7. RFM: Recency, Frequency, Monetary value

In [ ]:
customer_frequency = product_df.groupby("CustomerID")["InvoiceNo"].nunique()
#print(customer_frequency)

customer_monetary = product_df.groupby("CustomerID")["Revenue"].sum()
#print(customer_monetary)

customer_recency = product_df.groupby("CustomerID")["InvoiceDate"].max()
#print(customer_recency)

reference_date = product_df["InvoiceDate"].max()
#print(reference_date)

customer_recency_days = (reference_date - customer_recency)
print(customer_recency_days)

In [ ]:
customer_recency_days = customer_recency_days.dt.days
print(customer_recency_days)

In [ ]:
rfm = pd.DataFrame({
    "Recency": customer_recency_days,
    "Frequency": customer_frequency,
    "Monetary": customer_monetary
})

rfm.head()

In [ ]:
print(rfm["Monetary"].sort_values(ascending=False).head(10))

In [ ]:
print(rfm["Frequency"].sort_values(ascending=False).head(10))

In [ ]:
print(rfm["Recency"].sort_values().head(10))

In [ ]:
rfm.describe()

## 8. Looking at the RFM distributions

In [ ]:
rfm["Monetary"].plot(kind="hist")
plt.ylabel("Number of Customers")
plt.xlabel("Revenue")
plt.show()

In [ ]:
rfm["Frequency"].plot(kind="hist")
plt.ylabel("Number of Customers")
plt.xlabel("Numbers of orders")
plt.show()

## 9. First K-Means analysis

Scaling the original RFM features and exploring five clusters.

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

In [ ]:
rfm_scaled[:5]

In [ ]:
inertia = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.plot(range(2, 11), inertia)
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)
rfm.head()

In [ ]:
print(rfm.groupby("Cluster").mean())

In [ ]:
print(rfm["Cluster"].value_counts())

## 10. Log transformation and customer segmentation

Revisiting the analysis after reducing the skew in RFM features.

In [ ]:
rfm_log = np.log1p(rfm[["Recency", "Frequency", "Monetary"]])
rfm_log.head()

In [ ]:
scaler = StandardScaler()
rfm_log_scaled = scaler.fit_transform(rfm_log)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
rfm["Cluster_Log"] = kmeans.fit_predict(rfm_log_scaled)

In [ ]:
inertia_log = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_log_scaled)
    inertia_log.append(kmeans.inertia_)
    
rfm["Cluster_Log"].value_counts()

In [ ]:
plt.plot(range(2,11), inertia_log)
plt.ylabel("Inertia")
plt.xlabel("Number of clusters")
plt.title("Elbow")
plt.show()

In [ ]:
#print(rfm.groupby("Cluster_Log")[["Monetary", "Recency", "Frequency"]].mean())
print(rfm["Cluster_Log"].value_counts())

## 11. Cluster visualizations

In [ ]:
mon_avg = rfm.groupby("Cluster_Log")["Monetary"].mean()
mon_avg.plot(kind="bar")
plt.ylabel("Average Monetary Value")
plt.xlabel("Cluster")
plt.title("Average Monetary value per Cluster")
plt.show()

In [ ]:
cust_p = rfm["Cluster_Log"].value_counts()
cust_p.plot(kind="bar")
plt.ylabel("Number of Customers")
plt.xlabel("Cluster")
plt.title("Customers per Cluster")
plt.show()

## Cluster interpretation

The final interpretations from the project discussion:

| Cluster | Interpretation |
|---|---|
| 0 | Recent / lower-value |
| 1 | Loyal / high-value |
| 2 | Regular / promising |
| 3 | Becoming inactive |
| 4 | Inactive / low-value |

These names describe customer behavior; they are not predictions.

## Data source

Daqing Chen (2015), *Online Retail*, UCI Machine Learning Repository. [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33). [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
